<a href="https://colab.research.google.com/github/andreanchalvari/SkripsiAndre/blob/main/TRANSFORMER_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TRAINING TRANSFORMER

In [ ]:
import numpy as np
import tensorflow as tf

from tqdm.keras import TqdmCallback

print("=== TRAINING TRANSFORMER ===")

# LOAD DATASET
images = np.load(
    "/content/drive/MyDrive/hasil_transformer_64/images_balance_64.npy"
)

labels = np.load(
    "/content/drive/MyDrive/hasil_transformer_64/labels_balance_64.npy"
)

train_idx = np.load(
    "/content/drive/MyDrive/hasil_transformer_64/train_idx_64.npy"
)

test_idx = np.load(
    "/content/drive/MyDrive/hasil_transformer_64/test_idx_64.npy"
)

# AMBIL DATA TRAIN DAN TEST
x_train = images[train_idx]

y_train = labels[train_idx]

x_test = images[test_idx]

y_test = labels[test_idx]

# INFO DATASET
print("\n========== DATASET ==========")

print("Train :", x_train.shape)
print("Test  :", x_test.shape)

print("\nDistribusi TRAIN")

print(
    "Cumulus :",
    np.sum(y_train == 0)
)

print(
    "Stratus :",
    np.sum(y_train == 1)
)

print("\nDistribusi TEST")

print(
    "Cumulus :",
    np.sum(y_test == 0)
)

print(
    "Stratus :",
    np.sum(y_test == 1)
)

x_train = x_train.astype(np.float32)

x_test = x_test.astype(np.float32)

# CALLBACK
checkpoint_path = (
    "/content/drive/MyDrive/"
    "hasil_transformer_64/"
    "model_transformer_best_64.keras"
)

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(

        checkpoint_path,

        monitor="val_accuracy",

        save_best_only=True,

        verbose=1

    ),

    tf.keras.callbacks.EarlyStopping(

        monitor="val_loss",

        patience=10,

        restore_best_weights=True

    ),

    tf.keras.callbacks.ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=5,

        verbose=1

    ),

    TqdmCallback(verbose=1)

]

# TRAINING
print("\n🔥 MULAI TRAINING\n")

history = model.fit(

    x_train,

    y_train,

    validation_data=(

        x_test,

        y_test

    ),

    epochs=100,

    batch_size=8,

    callbacks=callbacks,

    verbose=0

)

print("\n✔ TRAINING SELESAI")

# SAVE MODEL
model.save(

    "/content/drive/MyDrive/"
    "hasil_transformer_64/"
    "model_transformer_final_64.keras"

)

# EVALUASI
loss, acc = model.evaluate(

    x_test,

    y_test,

    verbose=0

)

print(

    f"\n🔥 TEST ACCURACY : {acc*100:.2f}%"

)

print(

    f"TEST LOSS : {loss:.4f}"

)

# TESTING TRANSFORMER

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.utils import custom_object_scope

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

import seaborn as sns
import matplotlib.pyplot as plt

print("=== TESTING TRANSFORMER ===")

# CUSTOM LAYER
class Patches(layers.Layer):

    def __init__(self, patch_size=8, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1,self.patch_size,self.patch_size,1],
            strides=[1,self.patch_size,self.patch_size,1],
            rates=[1,1,1,1],
            padding="VALID"
        )

        patch_dims = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [batch_size,-1,patch_dims]
        )

        return patches

    def get_config(self):

        config = super().get_config()

        config.update({
            "patch_size": self.patch_size
        })

        return config


class PatchEncoder(layers.Layer):

    def __init__(
        self,
        num_patches,
        projection_dim,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.num_patches = num_patches
        self.projection_dim = projection_dim

        self.projection = layers.Dense(
            projection_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patch):

        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1
        )

        return (
            self.projection(patch)
            +
            self.position_embedding(positions)
        )

    def get_config(self):

        config = super().get_config()

        config.update({

            "num_patches": self.num_patches,

            "projection_dim": self.projection_dim

        })

        return config

# LOAD MODEL
with custom_object_scope({

    "Patches": Patches,

    "PatchEncoder": PatchEncoder

}):

    model = tf.keras.models.load_model(

        "/content/drive/MyDrive/hasil_transformer_64/model_transformer_final_64.keras",

        compile=False

    )

model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

print("✔ Model berhasil di-load")

# LOAD DATASET
images = np.load(
    "/content/drive/MyDrive/hasil_transformer_64/images_balance_64.npy"
)

labels = np.load(
    "/content/drive/MyDrive/hasil_transformer_64/labels_balance_64.npy"
)

test_idx = np.load(
    "/content/drive/MyDrive/hasil_transformer_64/test_idx_64.npy"
)

# DATA TEST
x_test = images[test_idx]

y_test = labels[test_idx]

print("\nJumlah data test :", len(x_test))

print(
    "Cumulus :",
    np.sum(y_test == 0)
)

print(
    "Stratus :",
    np.sum(y_test == 1)
)

# NORMALISASI
x_test = x_test.astype(np.float32)

# EVALUASI
loss, acc = model.evaluate(

    x_test,

    y_test,

    verbose=0

)

print(
    f"\n🔥 TEST ACCURACY : {acc*100:.2f}%"
)

print(
    f"TEST LOSS : {loss:.4f}"
)

# PREDIKSI
pred = model.predict(

    x_test,

    batch_size=8,

    verbose=1

)

y_pred = np.argmax(

    pred,

    axis=1

)

# CLASSIFICATION REPORT
print("\n========== CLASSIFICATION REPORT ==========\n")

print(

    classification_report(

        y_test,

        y_pred,

        target_names=[

            "Cumulus",

            "Stratus"

        ]

    )

)

# CONFUSION MATRIX
cm = confusion_matrix(

    y_test,

    y_pred

)

plt.figure(figsize=(6,5))

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues",

    xticklabels=[

        "Cumulus",

        "Stratus"

    ],

    yticklabels=[

        "Cumulus",

        "Stratus"

    ]

)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.tight_layout()

plt.show()